# Notebook 01 — Single-Atom Ω/Δ Scan

This notebook introduces a minimal two-level driven-atom model and scans excitation probability across:

- **Rabi frequency** $\Omega$
- **Detuning** $\Delta$

The goal is to build an interpretable first result for `rydberg-parameter-lab`.

We use a driven two-level Hamiltonian in the rotating frame:

$$
H = \frac{\Omega}{2}\sigma_x - \Delta |r\rangle\langle r|
$$

where $|g\rangle$ is the ground state and $|r\rangle$ is the excited / Rydberg-like state.


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qutip import basis, sigmax, mesolve


## Basis states and operators

In [ ]:
# Basis states
g = basis(2, 0)  # |g>
r = basis(2, 1)  # |r>

# Projector onto excited / Rydberg-like state
n_r = r * r.dag()

# Pauli-x operator in the {|g>, |r>} basis
sx = sigmax()


## Model definition

In [ ]:
def single_atom_hamiltonian(omega: float, delta: float):
    """Return the rotating-frame two-level Hamiltonian.

    H = (omega / 2) * sigma_x - delta * |r><r|
    """
    return 0.5 * omega * sx - delta * n_r


def excited_population(omega: float, delta: float, t_final: float = 10.0, n_steps: int = 400):
    """Simulate evolution from |g> and return final excited-state population."""
    H = single_atom_hamiltonian(omega, delta)
    psi0 = g
    times = np.linspace(0.0, t_final, n_steps)
    result = mesolve(H, psi0, times, [], [n_r])
    return float(np.real(result.expect[0][-1]))


## Quick sanity checks

In [ ]:
for omega, delta in [(1.0, 0.0), (1.0, 1.0), (2.0, 0.5)]:
    pop = excited_population(omega, delta)
    print(f"Omega={omega:.2f}, Delta={delta:.2f} -> final excited population = {pop:.4f}")

## 1D scan at fixed detuning

First, scan excitation probability versus $\Omega$ while holding detuning fixed.

In [ ]:
omegas = np.linspace(0.1, 5.0, 120)
delta_fixed = 0.5
pop_1d = [excited_population(omega, delta_fixed) for omega in omegas]

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(omegas, pop_1d)
plt.xlabel(r"Rabi frequency $\Omega$")
plt.ylabel(r"Final excited population $P_r$")
plt.title(rf"Single-atom scan at fixed detuning $\Delta={delta_fixed}$")
plt.tight_layout()
plt.show()

## 2D parameter scan

Now scan over both $(\Omega, \Delta)$ to produce a parameter landscape.

In [ ]:
omega_vals = np.linspace(0.1, 5.0, 70)
delta_vals = np.linspace(-3.0, 3.0, 70)

landscape = np.zeros((len(delta_vals), len(omega_vals)))

for i, delta in enumerate(delta_vals):
    for j, omega in enumerate(omega_vals):
        landscape[i, j] = excited_population(omega, delta)

In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    landscape,
    origin='lower',
    aspect='auto',
    extent=[omega_vals[0], omega_vals[-1], delta_vals[0], delta_vals[-1]],
)
plt.xlabel(r"Rabi frequency $\Omega$")
plt.ylabel(r"Detuning $\Delta$")
plt.title(r"Final excited-state population over $(\Omega, \Delta)$")
plt.colorbar(im, label=r"$P_r$")
plt.tight_layout()
plt.show()

## Interpretation

This notebook gives a minimal first view of parameter dependence:

- Near resonance ($\Delta \approx 0$), excitation is generally easier.
- Away from resonance, population transfer becomes less efficient.
- Increasing drive strength $\Omega$ can compensate for moderate detuning.

In later notebooks, this basic model can be extended to:

- three-level excitation schemes
- two-atom Rydberg blockade
- open-system Lindblad dynamics
- gate-fidelity landscapes and optimization


## Optional next step

A good follow-up is to replace the final-time population metric with:

- maximum population during evolution, or
- time-averaged excitation, or
- a fidelity metric for a target state.
